# aisuite local observability demo

This notebook shows the local viewer inside Jupyter. It has three paths:

1. **Real model + tools demo**: uses `OPENAI_API_KEY`, file tools, and live traces.
2. **Focused run viewer**: embeds only one run with `?embed=1&trace_id=...` so the notebook is not crowded.
3. **Scripted trace demo**: no API key; deterministic trace records for UI exploration.

The no-key path is useful for deterministic viewer demos, but the real-key path is the one to use when validating model/tool behavior.

In [ ]:
from IPython.display import IFrame, Markdown, display
from pathlib import Path
from types import SimpleNamespace
import json
import os
import tempfile

import aisuite as ai
from aisuite.framework.message import ChatCompletionMessageToolCall, Function, Message

## What to try

- Run the full viewer cell first so you can watch runs appear.
- Run the real OpenAI cell if `OPENAI_API_KEY` is set; it asks the model to inspect a tiny workspace using file tools.
- Run the focused viewer cell after each run to see the notebook-friendly single-run view.
- Run the scripted demo when you want a deterministic trace with tools and artifactized content.

## Start an in-memory viewer

The viewer is backed by `InMemoryTraceStore`, so no JSONL file is required. Agent runs write events through `viewer.trace_sink`.

In [ ]:
store = ai.tracing.InMemoryTraceStore()
viewer = ai.tracing.start_viewer(None, port=0, trace_store=store)
viewer.url

In [ ]:
display(IFrame(viewer.url, width='100%', height=720))

## Real OpenAI model + tools demo

This creates a tiny temporary workspace and gives the model read-only file tools. The prompt explicitly asks the model to use `list_files` and `read_file_lines`, so the run should include model events, tool events, and a final response.

In [ ]:
real_workspace = Path(tempfile.mkdtemp(prefix='aisuite-real-demo-'))
(real_workspace / 'README.md').write_text(
    '# Notebook Demo\n\nThis workspace exists so the model can inspect files through aisuite tools.\n',
    encoding='utf-8',
)
(real_workspace / 'todo.txt').write_text(
    '1. Show a real tool call\n2. Render the focused viewer\n3. Keep the demo compact\n',
    encoding='utf-8',
)
real_workspace

In [ ]:
if not os.getenv('OPENAI_API_KEY'):
    display(Markdown('`OPENAI_API_KEY` is not set. Set it and rerun this cell for the real model demo.'))
else:
    real_agent = ai.Agent(
        name='notebook_real_tools_demo',
        model='openai:gpt-4o-mini',
        instructions=(
            'You are demonstrating aisuite local observability. Use list_files first, '
            'then use read_file_lines on README.md, then answer concisely.'
        ),
        tools=ai.toolkits.files(root=real_workspace, allow_write=False),
        tags=['notebook', 'real-model', 'tools'],
        metadata={'demo': 'local_observability', 'mode': 'real_tools'},
    )
    real_result = ai.Runner.run_sync(
        real_agent,
        'Inspect this demo workspace with tools and summarize what the viewer should show.',
        trace_sinks=[viewer.trace_sink],
        run_name='real_openai_tools_demo',
        group_id='notebook-demo',
        max_turns=5,
    )
    display(Markdown(f'### Final output\n{real_result.final_output}'))
    display(IFrame(f'{viewer.url}?embed=1&trace_id={real_result.trace_id}&theme=dark', width='100%', height=720))

## Scripted no-key trace demo

This path does not call an LLM. It uses a scripted provider so the viewer can show predictable model, tool, approval, and artifact behavior without a key.

In [ ]:
def tool_call(name, arguments, call_id):
    return ChatCompletionMessageToolCall(
        id=call_id,
        type='function',
        function=Function(name=name, arguments=arguments),
    )

def response(content=None, tool_calls=None):
    message = Message(role='assistant', content=content, tool_calls=tool_calls)
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])

class ScriptedProvider:
    def __init__(self, responses):
        self.responses = list(responses)

    def chat_completions_create(self, *_args, **_kwargs):
        if not self.responses:
            raise RuntimeError('No scripted responses left.')
        return self.responses.pop(0)

workspace = Path(tempfile.mkdtemp(prefix='aisuite-notebook-demo-'))
(workspace / 'README.md').write_text('notebook demo workspace\n', encoding='utf-8')
large_content = 'notebook artifact line\n' * 1300

client = ai.Client()
client.providers['openai'] = ScriptedProvider([
    response(None, [tool_call('list_files', '{"path": "."}', 'call_list')]),
    response('I found a small demo workspace.'),
    response(None, [tool_call('write_file', json.dumps({'path': 'large.txt', 'content': large_content}), 'call_write')]),
    response('I wrote a large file so the viewer can show an artifactized argument.'),
])

def approve_all(context):
    return ai.ToolPolicyDecision(allowed=True, reason='approved by notebook demo')

scripted_agent = ai.Agent(
    name='notebook_scripted_demo',
    model='openai:gpt-4o-mini',
    instructions='Use tools to create a deterministic local observability demo.',
    tools=ai.toolkits.files(root=workspace, allow_write=True),
    tags=['notebook', 'scripted'],
    metadata={'demo': 'local_observability', 'mode': 'scripted'},
)

artifact_store = ai.FileArtifactStore(workspace / '.aisuite' / 'artifacts')
scripted_result = ai.Runner.run_sync(
    scripted_agent,
    'Inspect this workspace and write a large demo file.',
    client=client,
    trace_sinks=[viewer.trace_sink],
    tool_policy=approve_all,
    artifact_store=artifact_store,
    run_name='scripted_artifact_demo',
    group_id='notebook-demo',
    max_turns=5,
)

scripted_result.final_output

In [ ]:
display(IFrame(f'{viewer.url}?embed=1&trace_id={scripted_result.trace_id}&theme=dark', width='100%', height=720))

## Stop the viewer

Run this when you are done with the notebook server.

In [ ]:
# viewer.stop()